# [WIP] Proposed Redesign of SB-MOABB DataIO

In [1]:
%%capture
!pip install speechbrain moabb mne mne_bids

In [2]:
import json
import warnings
from pathlib import Path
from typing import Any

import mne
from mne_bids import BIDSPath, read_raw_bids
from moabb.datasets import BNCI2014_001
from moabb.datasets import download as dl
from moabb.datasets.base import BaseDataset as BaseMOABBDataset
from moabb.datasets.bids_interface import camel_to_kebab_case
from typing_extensions import Optional, Self

import speechbrain as sb
from speechbrain.dataio.dataset import DynamicItemDataset


class BaseEEGDataset(DynamicItemDataset):

    @classmethod
    def from_bids(
        cls,
        bids_path: Path | str | BIDSPath,
        json_path: str | Path,
        subjects=None,
        dynamic_items=[],
        output_keys=[],
    ) -> Self:
        if not isinstance(bids_path, BIDSPath):
            bids_path = BIDSPath(root=bids_path)
        json_data = cls.load_or_create_json_data_from_bids(
            bids_path, json_path, subjects=subjects
        )

        return cls(
            data=json_data, dynamic_items=dynamic_items, output_keys=output_keys
        )

    @classmethod
    def load_or_create_json_data_from_bids(
        cls,
        bids_path: BIDSPath,
        json_path: str | Path,
        subjects=None,
    ) -> dict[str, Any]:
        json_path = Path(json_path)
        if json_path.exists():
            with json_path.open() as fp:
                json_data = json.load(fp)
        else:
            json_data = cls.json_data_from_bids_path(bids_path)

            with json_path.open("w") as fp:
                json.dump(json_data, fp)

        if subjects is not None:
            json_data = {
                uid: data
                for uid, data in json_data.items()
                if data["subject"] in subjects
            }

        return json_data

    @classmethod
    def json_data_from_bids_path(cls, bids_path) -> dict[str, Any]:
        json_data = {}

        for path in bids_path.update(suffix="eeg").match(ignore_json=True):
            uid = path.fpath.name
            json_data[uid] = path.entities
            json_data[uid]["fpath"] = str(path.fpath)
        return json_data

    @classmethod
    def from_moabb(
        cls,
        dataset: BaseMOABBDataset,
        json_path: str | Path,
        subjects=None,
        save_path: Optional[str] = None,
        dynamic_items=[],
        output_keys=[],
    ) -> Self:
        json_path = Path(json_path)
        if json_path.exists():
            with json_path.open() as fp:
                json_data = json.load(fp)

            return cls(
                data=json_data,
                dynamic_items=dynamic_items,
                output_keys=output_keys,
            )

        mne_path = Path(dl.get_dataset_path(dataset.code, save_path))
        cache_dir = f"MNE-BIDS-{camel_to_kebab_case(dataset.code)}"
        cache_path = mne_path / cache_dir

        old_level = mne.set_log_level(verbose=False, return_old_level=True)
        with warnings.catch_warnings():
            warnings.simplefilter(action="ignore")
            for sub in (
                subjects if subjects is not None else dataset.subject_list
            ):
                dataset.get_data(
                    subjects=[sub],
                    cache_config=dict(use=True, save_raw=True, path=mne_path),
                )
        mne.set_log_level(old_level)

        return cls.from_bids(
            bids_path=cache_path,
            json_path=json_path,
            dynamic_items=dynamic_items,
            output_keys=output_keys,
            subjects=subjects,
        )


# TEST IT OUT
dataset = BaseEEGDataset.from_moabb(
    BNCI2014_001(),
    "data/MNE-BIDS-bnci2014-001.json",
    save_path="data",
    output_keys=["subject", "session", "fpath"],
)
dataset[0]

/home/drew-wagner/Research/benchmarks/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'subject': '1',
 'session': '0train',
 'fpath': 'data/MNE-BIDS-bnci2014-001/sub-1/ses-0train/eeg/sub-1_ses-0train_task-imagery_run-0_desc-c6ddd98f4a171af69d0c8f3a1a73f5b5_eeg.edf'}

In [3]:
from mne_bids import get_bids_path_from_fname

# HOW TO GET EPOCHS FROM RAW,
# based on MOABB code
dataset = BNCI2014_001()
bids_path = get_bids_path_from_fname(
    "data/MNE-BIDS-bnci2014-001/sub-1/ses-0train/eeg/sub-1_ses-0train_task-imagery_run-0_desc-c6ddd98f4a171af69d0c8f3a1a73f5b5_eeg.edf"
)
raw = read_raw_bids(bids_path, extra_params=dict(preload=False), verbose=0)
events, _ = mne.events_from_annotations(
    raw, event_id=dataset.event_id, verbose=False
)
offset = int(dataset.interval[0] * raw.info["sfreq"])
events[:, 0] -= offset
tmin = 0
tmax = dataset.interval[1] - dataset.interval[0]

epochs = mne.Epochs(
    raw,
    events,
    tmin=tmin,
    tmax=tmax,
    baseline=(tmin, tmax),
    event_id=dataset.event_id,
    verbose=False,
)
labels = epochs.events
epochs

<Epochs | 48 events (good & bad), 0 – 4 s (baseline 0 – 4 s), ~36 KiB, data not loaded,
 'left_hand': 12
 'right_hand': 12
 'feet': 12
 'tongue': 12>